Latest advancement in L2D: 2025: Adaptive alert prioritisation in security operations centres via learning to defer with human feedback


https://arxiv.org/pdf/2506.18462



In [1]:
import gym
from gym import spaces
import numpy as np

# Sigmoid function to compute probabilities for cross-entropy loss
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Cross-entropy loss function
def cross_entropy_loss(y_true, y_pred):
    # Clip predictions to avoid log(0)
    y_pred = np.clip(y_pred, 1e-10, 1 - 1e-10)
    return -np.sum(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# Define the Gym Environment
class L2DEnv(gym.Env):
    def __init__(self):
        super(L2DEnv, self).__init__()
        
        # Define the action space for the model (0 = do not defer, 1 = defer to human)
        self.action_space = spaces.Discrete(2)  # 0: Do not defer, 1: Defer to human
        
        # Define the observation space
        # The state will represent the model's data (X) and the human's data (Z)
        # For simplicity, assume 3 features for X (model's data) and 2 features for Z (human's data)
        self.observation_space = spaces.Box(low=0, high=1, shape=(5,), dtype=np.float32)
        
        # Initialize environment variables
        self.state = np.random.rand(5)  # Random initial state [model data, human data]
        self.done = False
        self.steps = 0
        self.confidence_threshold = 0.5  # Confidence threshold to decide whether to defer or not
    
    def reset(self):
        """Reset the environment to an initial state."""
        self.state = np.random.rand(5)  # Random starting point (X, Z)
        self.done = False
        self.steps = 0
        return self.state
    
    def model_prediction(self, X):
        """Calculate the model's prediction PM(Y = 1 | X)."""
        return sigmoid(np.dot(X[:3], np.random.randn(3)))  # Model uses first 3 features for prediction
    
    def human_prediction(self, X, Z):
        """Calculate the human's prediction PD(Y = 1 | X, Z)."""
        return sigmoid(np.dot(X[:3], np.random.randn(3)) + np.dot(Z, np.random.randn(2)))  # Human uses X and Z
    
    def decision_to_defer(self, X):
        """Calculate the decision to defer (s), based on the model's confidence."""
        confidence = self.model_prediction(X)
        return 1 if confidence < self.confidence_threshold else 0  # Defer if confidence is low
    
    def Pdef_er(self, Y, X, Z):
        """Compute the likelihood of the decision."""
        s = self.decision_to_defer(X)  # Get the decision to defer based on state X
        PM = self.model_prediction(X)  # Model prediction PM(Y = 1 | X)
        PD = self.human_prediction(X, Z)  # Human prediction PD(Y = 1 | X, Z)
        
        # Calculate the likelihood for each instance i (cross-entropy for model vs. human)
        likelihood = (1 - s) * (Y * np.log(PM) + (1 - Y) * np.log(1 - PM)) + s * (Y * np.log(PD) + (1 - Y) * np.log(1 - PD))
        return likelihood
    
    def Ldef_er(self, Y, X, Z):
        """Compute the loss function."""
        s = self.decision_to_defer(X)  # Get the decision to defer based on state X
        PM = self.model_prediction(X)  # Model prediction PM(Y = 1 | X)
        PD = self.human_prediction(X, Z)  # Human prediction PD(Y = 1 | X, Z)
        
        # Cross-entropy loss for each case: either the model's or human's prediction based on s
        model_loss = cross_entropy_loss(Y, PM)  # Loss using model's prediction
        human_loss = cross_entropy_loss(Y, PD)  # Loss using human's prediction
        
        # Combine the two losses based on s
        return (1 - s) * model_loss + s * human_loss  # Weighted loss
    
    def step(self, action):
        """Define one step of interaction in the environment."""
        self.steps += 1
        reward = 0
        
        # Simulate some random ground truth Y (binary values)
        Y = np.random.randint(0, 2, size=1)  # Random ground truth (0 or 1)
        
        # Get predictions for the model and human
        s = self.decision_to_defer(self.state)  # Get the defer decision based on the model’s confidence
        PM = self.model_prediction(self.state)  # Model prediction
        PD = self.human_prediction(self.state[:3], self.state[3:])  # Human prediction (split X and Z)
        
        # Reward based on model's deferral decision and loss
        if action == 0:  # Model does not defer (takes the decision itself)
            if s == 0:  # Model is confident
                reward = 1 if (Y == (PM >= 0.5)) else -1  # Correct if model prediction matches ground truth
            else:  # Model should defer but doesn't
                reward = -1
        elif action == 1:  # Model defers to human (s = 1)
            if s == 1:  # Model defers, human makes the decision
                reward = 1 if (Y == (PD >= 0.5)) else -1  # Correct if human prediction matches ground truth
            else:  # Model should act, but defers
                reward = -1
        
        # Check if the episode should end
        if self.steps >= 10:  # Limit steps for this example
            self.done = True
        
        # Return the next state, reward, done flag, and additional info (if any)
        return self.state, reward, self.done, {}
    
    def render(self, mode='human'):
        """Render the environment state for visualization."""
        print(f"Step: {self.steps}, State: {self.state}, Confidence: {self.model_prediction(self.state)}, Done: {self.done}")



In [2]:
# Create the environment
env = L2DEnv()

# Initialize environment
state = env.reset()

done = False

# Run a few steps in the environment
while not done:
    action = np.random.choice([0, 1])  # Random action for testing (you can replace this with a model or policy)
    print(f"Action: {action}")  # Display the action chosen
    state, reward, done, _ = env.step(action)
    env.render()  # Print the state after each action

Action: 1
Step: 1, State: [0.70701987 0.6377545  0.38092579 0.95028826 0.53697848], Confidence: 0.714694929457614, Done: False
Action: 0
Step: 2, State: [0.70701987 0.6377545  0.38092579 0.95028826 0.53697848], Confidence: 0.7458853904545596, Done: False
Action: 1
Step: 3, State: [0.70701987 0.6377545  0.38092579 0.95028826 0.53697848], Confidence: 0.45523976106893943, Done: False
Action: 1
Step: 4, State: [0.70701987 0.6377545  0.38092579 0.95028826 0.53697848], Confidence: 0.723848214542226, Done: False
Action: 1
Step: 5, State: [0.70701987 0.6377545  0.38092579 0.95028826 0.53697848], Confidence: 0.7145044663861609, Done: False
Action: 0
Step: 6, State: [0.70701987 0.6377545  0.38092579 0.95028826 0.53697848], Confidence: 0.24284390399273842, Done: False
Action: 0
Step: 7, State: [0.70701987 0.6377545  0.38092579 0.95028826 0.53697848], Confidence: 0.5185165452705778, Done: False
Action: 0
Step: 8, State: [0.70701987 0.6377545  0.38092579 0.95028826 0.53697848], Confidence: 0.722637